# SensorGuard - Anomaly Detection in IoT Sensor Data

End-to-end notebook: data generation, preprocessing, feature engineering,
anomaly detection (Z-Score, Isolation Forest, One-Class SVM), root cause analysis, and evaluation.


## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='darkgrid')
%matplotlib inline

DATA_PATH  = os.path.join('..', 'data', 'sensor_data.csv')
OUTPUT_DIR = os.path.join('..', 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

RAW_FEATURES = ['temperature', 'pressure', 'vibration', 'humidity', 'voltage']
NORM_COLS    = [f'{c}_norm' for c in RAW_FEATURES]
print('Setup complete.')

## 1. Generate Synthetic IoT Dataset

Simulates 2000 minutes of sensor readings from 5 sensors with ~4% injected anomalies.


In [ ]:
from generate_data import generate

df_raw = generate(n=2000, seed=42)
df_raw.to_csv(DATA_PATH, index=False)

print(f'Shape: {df_raw.shape}')
print(f'Anomalies (ground truth): {df_raw.label.sum()}')
df_raw.head()

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)
for ax, col in zip(axes, RAW_FEATURES):
    ax.plot(df_raw['timestamp'], df_raw[col], linewidth=0.7)
    ax.set_ylabel(col)
axes[0].set_title('Raw Sensor Readings', fontsize=13, fontweight='bold')
axes[-1].set_xlabel('Time')
plt.tight_layout()
plt.show()

## 2. Preprocessing

- Forward-fill missing values
- Min-Max normalize each sensor to [0, 1]


In [ ]:
from preprocessing import preprocess

df, scaler = preprocess(DATA_PATH, RAW_FEATURES)
print('Missing values after cleaning:', df[RAW_FEATURES].isnull().sum().sum())
df[NORM_COLS].describe().round(3)

## 3. Feature Engineering

Creates rolling mean, rolling std, trend (diff), and rate-of-change for each sensor.


In [ ]:
from feature_engineering import engineer_features

df = engineer_features(df, NORM_COLS)
print(f'Total columns after feature engineering: {len(df.columns)}')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
ax1.plot(df['timestamp'], df['temperature_norm'], label='Normalized', linewidth=0.7)
ax1.plot(df['timestamp'], df['temperature_norm_roll_mean'], label='Rolling Mean', linewidth=1.2)
ax1.set_ylabel('Value'); ax1.legend()
ax2.plot(df['timestamp'], df['temperature_norm_roll_std'], color='green', linewidth=0.8, label='Rolling Std')
ax2.set_ylabel('Std Dev'); ax2.legend()
ax1.set_title('Rolling Features - Temperature', fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Anomaly Detection

### 4a. Z-Score
Flags readings more than 3 standard deviations from the mean.


In [ ]:
from anomaly_detection import (
    zscore_detection, isolation_forest_detection,
    ocsvm_detection, combine_anomalies
)

eng_cols = [c for c in df.columns if any(
    c.endswith(s) for s in ['_norm', '_roll_mean', '_roll_std', '_trend', '_roc']
)]

df = zscore_detection(df, eng_cols)
print('Z-Score anomalies:', df.zscore_anomaly.sum())

### 4b. Isolation Forest
Tree-based model that isolates outliers faster than normal points.


In [ ]:
df = isolation_forest_detection(df, eng_cols, contamination=0.05)
print('Isolation Forest anomalies:', df.iforest_anomaly.sum())

### 4c. One-Class SVM (Bonus)
Learns a tight boundary around normal data; points outside are anomalies.


In [ ]:
df = ocsvm_detection(df, eng_cols, nu=0.05)
print('One-Class SVM anomalies:', df.ocsvm_anomaly.sum())

### 4d. Ensemble - combine all three methods


In [ ]:
df = combine_anomalies(df)
print('Total ensemble anomalies:', df.anomaly.sum())

## 5. Visualisation


In [ ]:
fig, axes = plt.subplots(len(RAW_FEATURES), 1, figsize=(14, 14), sharex=True)
fig.suptitle('SensorGuard - Sensor Readings with Anomalies', fontsize=14, fontweight='bold')
for ax, col in zip(axes, RAW_FEATURES):
    ax.plot(df['timestamp'], df[col], color='steelblue', linewidth=0.7, label=col)
    anom = df[df.anomaly]
    ax.scatter(anom['timestamp'], anom[col], color='red', s=18, zorder=5, label='Anomaly')
    ax.set_ylabel(col); ax.legend(loc='upper right', fontsize=8)
axes[-1].set_xlabel('Time')
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df['timestamp'], df['iforest_score'], color='darkorange', linewidth=0.8, label='IF Score')
ax.axhline(0, color='red', linestyle='--', linewidth=1, label='Decision boundary')
anom = df[df.anomaly]
ax.scatter(anom['timestamp'], anom['iforest_score'], color='red', s=18, zorder=5)
ax.set_title('Isolation Forest Anomaly Score', fontweight='bold')
ax.set_xlabel('Time'); ax.set_ylabel('Score (lower = more anomalous)')
ax.legend(); plt.tight_layout(); plt.show()

## 6. Root Cause Analysis


In [ ]:
from root_cause import feature_contribution, rank_culprits

corr = df[RAW_FEATURES].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, ax=ax)
ax.set_title('Feature Correlation Matrix', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
anomaly_df = feature_contribution(df, RAW_FEATURES)
anomaly_df[['timestamp', 'top_culprit', 'max_deviation']].head(10)

In [ ]:
counts = anomaly_df['top_culprit'].value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
counts.plot(kind='bar', ax=ax, color='tomato', edgecolor='black')
ax.set_title('Root Cause: Feature Anomaly Contribution', fontweight='bold')
ax.set_xlabel('Sensor'); ax.set_ylabel('Count')
plt.tight_layout(); plt.show()
print(f'Top culprit: {counts.index[0]} ({counts.iloc[0]} anomalies)')

## 7. Evaluation (Precision / Recall / F1)

Compare predictions against the injected ground-truth labels.


In [ ]:
from evaluation import evaluate
evaluate(df)

## 8. Save Outputs


In [ ]:
df.to_csv(os.path.join(OUTPUT_DIR, 'processed_data.csv'), index=False)
df[df.anomaly].to_csv(os.path.join(OUTPUT_DIR, 'anomaly_results.csv'), index=False)
print('Saved processed_data.csv and anomaly_results.csv to outputs/')

---
## Summary

| Step | Method | Notes |
|------|--------|-------|
| Detection | Z-Score (threshold=3) | Statistical outliers |
| Detection | Isolation Forest | ML-based isolation |
| Detection | One-Class SVM | Boundary-based |
| Root Cause | Feature deviation ranking | Top culprit sensor |
| Evaluation | Precision / Recall / F1 | vs ground-truth labels |
